# Memory Management & Compression - dask_setup Recipe

This notebook demonstrates advanced memory management features in dask_setup, including memory spilling thresholds, compression algorithms, and monitoring techniques for optimal performance on memory-constrained systems.

## Key Learning Points
- Memory spilling thresholds and when they trigger
- Impact of different compression algorithms
- Monitoring worker memory usage in real-time
- Optimal configurations for memory-constrained environments
- HPC storage management with spill directories

## Setup and Imports

Import necessary libraries and configure the environment for memory monitoring.

In [ ]:
import sys
import time
import threading
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# Add dask_setup to path
sys.path.insert(0, str(Path.cwd().parent.parent.parent / "src"))
sys.path.insert(0, str(Path.cwd().parent))

# Core imports
import numpy as np
import dask
import dask.array as da

# Import dask_setup
try:
    from dask_setup import setup_dask_client
    print("✅ dask_setup imported successfully")
except ImportError as e:
    print(f"❌ Failed to import dask_setup: {e}")
    
# System monitoring imports
try:
    import psutil
    PSUTIL_AVAILABLE = True
    print("✅ psutil available for system monitoring")
except ImportError:
    PSUTIL_AVAILABLE = False
    print("⚠️  psutil not available - system monitoring limited")

# Utility imports
try:
    from utils import format_duration, format_bytes
    print("✅ utils imported successfully")
except ImportError as e:
    print(f"⚠️  utils not available: {e}")
    # Define minimal versions
    def format_duration(seconds):
        return f"{seconds:.2f}s"
    def format_bytes(bytes_val):
        return f"{bytes_val / (1024**2):.1f} MB"

# Visualization imports
try:
    import matplotlib.pyplot as plt
    PLOTTING_AVAILABLE = True
    print("✅ matplotlib available for plotting")
except ImportError:
    PLOTTING_AVAILABLE = False
    print("⚠️  matplotlib not available - plotting disabled")

print("\n🚀 Setup complete!")

## System Memory Information

Let's examine the system's memory configuration before we start our tests.

In [ ]:
import os

print("💻 SYSTEM MEMORY ANALYSIS")
print("=" * 50)

if PSUTIL_AVAILABLE:
    # System memory
    mem = psutil.virtual_memory()
    swap = psutil.swap_memory()
    
    print(f"Physical Memory:")
    print(f"   Total: {mem.total / (1024**3):.2f} GB")
    print(f"   Available: {mem.available / (1024**3):.2f} GB")
    print(f"   Used: {mem.used / (1024**3):.2f} GB ({mem.percent:.1f}%)")
    print(f"   Free: {mem.free / (1024**3):.2f} GB")
    
    print(f"\nSwap Memory:")
    print(f"   Total: {swap.total / (1024**3):.2f} GB")
    print(f"   Used: {swap.used / (1024**3):.2f} GB ({swap.percent:.1f}%)")
    print(f"   Free: {swap.free / (1024**3):.2f} GB")
    
    # CPU information
    cpu_count = psutil.cpu_count(logical=False)
    cpu_logical = psutil.cpu_count(logical=True)
    
    print(f"\nCPU Information:")
    print(f"   Physical cores: {cpu_count}")
    print(f"   Logical cores: {cpu_logical}")
    
else:
    print("⚠️  System monitoring not available (psutil required)")

# Check environment variables for HPC
print(f"\n🖥️  HPC ENVIRONMENT VARIABLES:")
hpc_vars = {
    'PBS_JOBFS': 'Fast local storage directory',
    'TMPDIR': 'Temporary directory',
    'PBS_MEM': 'Allocated memory',
    'SLURM_MEM_PER_NODE': 'Memory per node (SLURM)'
}

hpc_detected = False
for var, description in hpc_vars.items():
    value = os.environ.get(var)
    if value:
        print(f"   {var}: {value} ({description})")
        hpc_detected = True

if not hpc_detected:
    print("   No HPC environment variables detected (running locally)")

print("\n✅ System analysis complete")

## Memory Monitoring Functions

Let's create functions to monitor memory usage and spill statistics during computations.

In [ ]:
def get_worker_memory_stats(client):
    """Get detailed memory statistics from all workers."""
    
    def worker_memory_info():
        """Function to run on each worker to collect memory stats."""
        import gc
        import os
        try:
            import psutil
            
            # Get process memory info
            process = psutil.Process()
            mem_info = process.memory_info()
            
            # Get memory percentage
            mem_percent = process.memory_percent()
            
            # Check temporary directory usage
            tmpdir = os.environ.get('TMPDIR', '/tmp')
            tmpdir_info = {}
            
            try:
                if os.path.exists(tmpdir):
                    disk_usage = psutil.disk_usage(tmpdir)
                    tmpdir_info = {
                        'tmpdir_path': tmpdir,
                        'tmpdir_total_gb': disk_usage.total / (1024**3),
                        'tmpdir_used_gb': disk_usage.used / (1024**3),
                        'tmpdir_free_gb': disk_usage.free / (1024**3),
                        'tmpdir_percent': (disk_usage.used / disk_usage.total) * 100
                    }
            except Exception as e:
                tmpdir_info['tmpdir_error'] = str(e)
            
            # Force garbage collection and get stats
            collected = gc.collect()
            
            return {
                'memory_rss_mb': mem_info.rss / (1024**2),
                'memory_vms_mb': mem_info.vms / (1024**2),
                'memory_percent': mem_percent,
                'gc_collected': collected,
                **tmpdir_info
            }
            
        except ImportError:
            return {'error': 'psutil not available on worker'}
        except Exception as e:
            return {'error': str(e)}
    
    try:
        # Run on all workers
        worker_stats = client.run(worker_memory_info)
        
        # Calculate aggregates
        total_rss = 0
        total_vms = 0
        total_tmpdir_used = 0
        total_tmpdir_free = 0
        valid_workers = 0
        
        for worker_id, stats in worker_stats.items():
            if 'error' not in stats:
                total_rss += stats.get('memory_rss_mb', 0)
                total_vms += stats.get('memory_vms_mb', 0)
                total_tmpdir_used += stats.get('tmpdir_used_gb', 0)
                total_tmpdir_free += stats.get('tmpdir_free_gb', 0)
                valid_workers += 1
        
        return {
            'worker_count': len(worker_stats),
            'valid_workers': valid_workers,
            'total_memory_rss_mb': total_rss,
            'total_memory_vms_mb': total_vms,
            'average_memory_rss_mb': total_rss / max(valid_workers, 1),
            'total_tmpdir_used_gb': total_tmpdir_used,
            'total_tmpdir_free_gb': total_tmpdir_free,
            'individual_workers': worker_stats
        }
        
    except Exception as e:
        return {'error': str(e)}


def monitor_computation(client, computation_func, description="Computation"):
    """Monitor memory usage during a computation with timeline tracking."""
    
    print(f"\n📊 Monitoring: {description}")
    print("-" * 40)
    
    memory_timeline = []
    start_time = time.perf_counter()
    
    # Initial memory snapshot
    initial_stats = get_worker_memory_stats(client)
    memory_timeline.append({
        'time_elapsed': 0.0,
        'stage': 'initial',
        **initial_stats
    })
    
    print(f"Initial memory usage: {initial_stats.get('total_memory_rss_mb', 0):.1f} MB across {initial_stats.get('valid_workers', 0)} workers")
    
    # Set up computation monitoring
    result = None
    computation_done = threading.Event()
    monitor_interval = 1.0  # Monitor every second
    
    def run_computation():
        nonlocal result
        try:
            result = computation_func()
            computation_done.set()
        except Exception as e:
            print(f"❌ Computation failed: {e}")
            result = None
            computation_done.set()
    
    def monitor_memory():
        while not computation_done.is_set():
            time.sleep(monitor_interval)
            if computation_done.is_set():
                break
                
            elapsed = time.perf_counter() - start_time
            stats = get_worker_memory_stats(client)
            
            memory_timeline.append({
                'time_elapsed': elapsed,
                'stage': 'computing',
                **stats
            })
            
            # Print progress
            rss_mb = stats.get('total_memory_rss_mb', 0)
            tmpdir_used = stats.get('total_tmpdir_used_gb', 0)
            print(f"  {elapsed:6.1f}s: Memory={rss_mb:6.1f}MB, Spill={tmpdir_used:5.1f}GB")
    
    # Start both threads
    comp_thread = threading.Thread(target=run_computation)
    monitor_thread = threading.Thread(target=monitor_memory)
    
    comp_thread.start()
    monitor_thread.start()
    
    # Wait for completion
    comp_thread.join()
    computation_done.set()
    monitor_thread.join()
    
    # Final memory snapshot
    final_time = time.perf_counter() - start_time
    final_stats = get_worker_memory_stats(client)
    memory_timeline.append({
        'time_elapsed': final_time,
        'stage': 'final',
        **final_stats
    })
    
    print(f"Final memory usage: {final_stats.get('total_memory_rss_mb', 0):.1f} MB")
    print(f"Total spill usage: {final_stats.get('total_tmpdir_used_gb', 0):.1f} GB")
    print(f"Computation time: {final_time:.2f}s")
    
    return result, memory_timeline

print("✅ Memory monitoring functions defined")

## Memory Pressure Test

Let's create a computation that will cause memory pressure and spilling to demonstrate the monitoring.

In [ ]:
# Create a Dask client with aggressive memory settings to trigger spilling
client, cluster, temp_dir = setup_dask_client(
    workload_type="cpu",
    max_workers=2,  # Fewer workers = more memory pressure per worker
    reserve_mem_gb=15.0,  # Reserve less memory to create pressure
    dashboard=True
)

print(f"✅ Cluster created with {len(client.scheduler_info()['workers'])} workers")
print(f"   Dashboard: {client.dashboard_link}")
print(f"   Temp directory: {temp_dir}")

# Display worker configuration
worker_info = client.scheduler_info()['workers']
if worker_info:
    first_worker = next(iter(worker_info.values()))
    memory_limit = first_worker.get('memory_limit', 0)
    print(f"   Memory per worker: {memory_limit / (1024**3):.1f} GB")
    print(f"   Threads per worker: {first_worker.get('nthreads', 1)}")

# Get initial cluster stats
initial_stats = get_worker_memory_stats(client)
print(f"\n📊 Initial cluster state:")
print(f"   Workers responding: {initial_stats.get('valid_workers', 0)}/{initial_stats.get('worker_count', 0)}")
print(f"   Total memory usage: {initial_stats.get('total_memory_rss_mb', 0):.1f} MB")
print(f"   Temp directory usage: {initial_stats.get('total_tmpdir_used_gb', 0):.1f} GB")

In [ ]:
def create_memory_pressure_computation(size_gb=2.0):
    """Create a computation that will cause memory pressure and spilling."""
    
    print(f"\n🔥 Creating memory pressure computation ({size_gb} GB target)")
    
    # Calculate array dimensions for approximately the target size
    elements_per_gb = (1024**3) // 8  # 8 bytes per float64
    total_elements = int(size_gb * elements_per_gb)
    
    # Create a cube-like shape that will stress memory
    cube_root = int(total_elements ** (1/3))
    shape = (cube_root, cube_root, cube_root)
    
    # Use moderate chunk sizes to force multiple chunks but allow spilling
    chunk_size = min(500, cube_root // 4)
    
    print(f"   Array shape: {shape}")
    print(f"   Chunk size: {chunk_size}")
    print(f"   Estimated size: {np.prod(shape) * 8 / (1024**3):.2f} GB")
    
    def heavy_computation():
        print("   Creating large arrays...")
        
        # Create multiple large arrays to force memory pressure
        x = da.random.random(shape, chunks=(chunk_size, chunk_size, chunk_size), dtype='float64')
        y = da.random.random(shape, chunks=(chunk_size, chunk_size, chunk_size), dtype='float64')
        
        print(f"   Arrays created. X chunks: {x.nchunks}, Y chunks: {y.nchunks}")
        
        # Perform operations that create intermediate arrays
        print("   Performing mathematical operations...")
        z = (x + y) * 2.0
        w = da.sin(z) + da.cos(z)
        
        print("   Computing final result...")
        # This reduction will force computation of all intermediate results
        result = w.sum().compute()
        
        return result
    
    return heavy_computation

# Create the computation function
computation = create_memory_pressure_computation(size_gb=1.5)

# Monitor the computation
result, memory_timeline = monitor_computation(
    client, 
    computation, 
    "Heavy computation with memory pressure"
)

if result is not None:
    print(f"\n✅ Computation succeeded!")
    print(f"   Result: {result:.6e}")
    print(f"   Timeline points: {len(memory_timeline)}")
else:
    print(f"\n❌ Computation failed")

## Memory Timeline Analysis

Let's analyze the memory usage timeline and visualize the spilling behavior.

In [ ]:
def analyze_memory_timeline(timeline):
    """Analyze memory usage timeline and extract key insights."""
    
    print("\n" + "=" * 60)
    print("📈 MEMORY TIMELINE ANALYSIS")
    print("=" * 60)
    
    if not timeline:
        print("❌ No timeline data available")
        return
    
    # Extract key metrics
    times = []
    memory_values = []
    spill_values = []
    
    for point in timeline:
        if 'error' not in point:
            times.append(point.get('time_elapsed', 0))
            memory_values.append(point.get('total_memory_rss_mb', 0))
            spill_values.append(point.get('total_tmpdir_used_gb', 0))
    
    if not times:
        print("❌ No valid timeline data")
        return
    
    # Calculate statistics
    initial_memory = memory_values[0] if memory_values else 0
    peak_memory = max(memory_values) if memory_values else 0
    final_memory = memory_values[-1] if memory_values else 0
    
    max_spill = max(spill_values) if spill_values else 0
    total_time = max(times) if times else 0
    
    print(f"⏱️  Duration: {total_time:.1f} seconds")
    print(f"💾 Memory Usage:")
    print(f"   Initial: {initial_memory:.1f} MB")
    print(f"   Peak: {peak_memory:.1f} MB")
    print(f"   Final: {final_memory:.1f} MB")
    print(f"   Growth: {peak_memory - initial_memory:.1f} MB")
    
    print(f"💿 Spill Usage:")
    print(f"   Maximum: {max_spill:.1f} GB")
    
    # Identify spilling events
    spill_events = 0
    for i in range(1, len(spill_values)):
        if spill_values[i] > spill_values[i-1] + 0.1:  # >100MB increase
            spill_events += 1
    
    print(f"   Spill events detected: {spill_events}")
    
    # Memory efficiency analysis
    if max_spill > 0:
        efficiency = (peak_memory / 1024) / (peak_memory / 1024 + max_spill)  # Ratio of memory to total
        print(f"   Memory efficiency: {efficiency:.1%} (memory vs memory+spill)")
    
    return {
        'duration': total_time,
        'initial_memory_mb': initial_memory,
        'peak_memory_mb': peak_memory,
        'final_memory_mb': final_memory,
        'max_spill_gb': max_spill,
        'spill_events': spill_events,
        'times': times,
        'memory_values': memory_values,
        'spill_values': spill_values
    }

# Analyze our computation timeline
if 'memory_timeline' in locals():
    analysis = analyze_memory_timeline(memory_timeline)
else:
    print("⚠️  No memory timeline available from previous computation")
    analysis = None

## Memory Usage Visualization

Let's create visualizations of memory usage and spilling over time.

In [ ]:
if PLOTTING_AVAILABLE and analysis:
    import matplotlib.pyplot as plt
    
    # Create subplots for memory usage and spill data
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    times = analysis['times']
    memory_values = analysis['memory_values']
    spill_values = analysis['spill_values']
    
    # Plot 1: Memory usage over time
    ax1.plot(times, memory_values, 'b-', linewidth=2, marker='o', markersize=4, label='Memory Usage (MB)')
    ax1.set_ylabel('Memory Usage (MB)', fontsize=12)
    ax1.set_title('Memory Usage and Spilling Timeline', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Highlight peak memory
    peak_idx = memory_values.index(max(memory_values))
    ax1.annotate(f'Peak: {max(memory_values):.1f} MB', 
                xy=(times[peak_idx], memory_values[peak_idx]),
                xytext=(10, 10), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
    
    # Plot 2: Spill usage over time
    ax2.plot(times, spill_values, 'r-', linewidth=2, marker='s', markersize=4, label='Spill Usage (GB)')
    ax2.set_xlabel('Time (seconds)', fontsize=12)
    ax2.set_ylabel('Spill Usage (GB)', fontsize=12)
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    # Highlight spilling events
    if max(spill_values) > 0:
        max_spill_idx = spill_values.index(max(spill_values))
        ax2.annotate(f'Max Spill: {max(spill_values):.1f} GB', 
                    xy=(times[max_spill_idx], spill_values[max_spill_idx]),
                    xytext=(10, 10), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='red', alpha=0.7),
                    arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
    
    # Add background shading for computation phases
    if len(times) > 2:
        ax1.axvspan(times[1], times[-2], alpha=0.1, color='green', label='Active Computation')
        ax2.axvspan(times[1], times[-2], alpha=0.1, color='green', label='Active Computation')
    
    plt.tight_layout()
    plt.show()
    
    # Create a summary statistics plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    
    categories = ['Initial\nMemory', 'Peak\nMemory', 'Final\nMemory', 'Max\nSpill']
    values = [
        analysis['initial_memory_mb'],
        analysis['peak_memory_mb'], 
        analysis['final_memory_mb'],
        analysis['max_spill_gb'] * 1024  # Convert to MB for comparison
    ]
    colors = ['lightblue', 'orange', 'lightgreen', 'red']
    
    bars = ax.bar(categories, values, color=colors, alpha=0.7, edgecolor='black')
    ax.set_ylabel('Memory Usage (MB)', fontsize=12)
    ax.set_title('Memory Usage Summary', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        if value > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
                   f'{value:.0f}\nMB', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
else:
    if not PLOTTING_AVAILABLE:
        print("📊 Plotting not available (matplotlib not installed)")
        print("   Install matplotlib to see memory timeline visualizations")
    else:
        print("📊 No analysis data available for plotting")
    
    # Print text-based summary instead
    if analysis:
        print("\n📊 TEXT-BASED MEMORY SUMMARY:")
        print(f"   Timeline: 0s → {analysis['duration']:.1f}s")
        print(f"   Memory: {analysis['initial_memory_mb']:.1f} → {analysis['peak_memory_mb']:.1f} → {analysis['final_memory_mb']:.1f} MB")
        print(f"   Spill: {analysis['max_spill_gb']:.1f} GB maximum")
        print(f"   Events: {analysis['spill_events']} spill events detected")

## Compression Algorithm Comparison

Now let's test different compression algorithms to see their impact on performance and memory usage.

In [ ]:
# Clean up the previous cluster
client.close()
cluster.close()
print("🧹 Previous cluster cleaned up")

def test_compression_algorithm(compression_type, array_size_gb=1.0):
    """Test a specific compression algorithm with memory monitoring."""
    
    print(f"\n🧪 Testing compression: {compression_type}")
    print("-" * 40)
    
    try:
        # Create cluster with specific compression settings
        # Note: In real dask_setup, you might configure this through environment variables
        # or configuration files. Here we'll create a standard cluster.
        client, cluster, temp_dir = setup_dask_client(
            workload_type="cpu",
            max_workers=2,
            reserve_mem_gb=15.0,  # Create memory pressure
            dashboard=False  # Disable dashboard for cleaner output
        )
        
        print(f"   Cluster: {len(client.scheduler_info()['workers'])} workers")
        print(f"   Temp dir: {temp_dir}")
        
        # Configure compression via Dask config (if supported)
        try:
            import dask
            with dask.config.set({'distributed.worker.daemon': False,
                                 'distributed.comm.compression': compression_type}):
                
                # Create computation that will cause spilling
                computation = create_memory_pressure_computation(size_gb=array_size_gb)
                
                # Monitor the computation
                start_time = time.perf_counter()
                result, timeline = monitor_computation(
                    client, computation, f"Computation with {compression_type} compression"
                )
                total_time = time.perf_counter() - start_time
                
        except Exception as config_error:
            print(f"   ⚠️  Could not configure compression: {config_error}")
            # Fall back to standard computation
            computation = create_memory_pressure_computation(size_gb=array_size_gb)
            start_time = time.perf_counter()
            result, timeline = monitor_computation(
                client, computation, f"Computation (compression config failed)"
            )
            total_time = time.perf_counter() - start_time
        
        # Get final statistics
        final_stats = get_worker_memory_stats(client)
        
        # Clean up
        client.close()
        cluster.close()
        
        if result is not None:
            return {
                'compression': compression_type,
                'success': True,
                'computation_time': total_time,
                'result_value': float(result),
                'timeline': timeline,
                'final_memory_mb': final_stats.get('total_memory_rss_mb', 0),
                'max_spill_gb': final_stats.get('total_tmpdir_used_gb', 0)
            }
        else:
            return {
                'compression': compression_type,
                'success': False,
                'error': 'Computation returned None'
            }
            
    except Exception as e:
        print(f"   ❌ Test failed: {e}")
        return {
            'compression': compression_type,
            'success': False,
            'error': str(e)
        }


# Test different compression algorithms
compression_algorithms = ['lz4', 'zstd', 'snappy', 'auto']
compression_results = {}

print("🔬 COMPRESSION ALGORITHM BENCHMARK")
print("=" * 50)

for compression in compression_algorithms:
    try:
        result = test_compression_algorithm(compression, array_size_gb=0.8)  # Smaller size for demo
        compression_results[compression] = result
        
        if result['success']:
            print(f"✅ {compression:8}: {result['computation_time']:.2f}s, Memory: {result.get('final_memory_mb', 0):.1f}MB, Spill: {result.get('max_spill_gb', 0):.1f}GB")
        else:
            print(f"❌ {compression:8}: {result.get('error', 'Failed')}")
            
    except KeyboardInterrupt:
        print(f"\n🛑 Benchmark interrupted during {compression}")
        break
    except Exception as e:
        print(f"❌ {compression:8}: Unexpected error: {e}")
        compression_results[compression] = {
            'compression': compression,
            'success': False,
            'error': str(e)
        }

## Compression Results Analysis

Let's analyze the performance differences between compression algorithms.

In [ ]:
def analyze_compression_results(results):
    """Analyze compression benchmark results."""
    
    print("\n" + "=" * 60)
    print("📊 COMPRESSION ALGORITHM ANALYSIS")
    print("=" * 60)
    
    # Filter successful results
    successful = {k: v for k, v in results.items() if v.get('success', False)}
    failed = {k: v for k, v in results.items() if not v.get('success', False)}
    
    if not successful:
        print("❌ No successful compression tests to analyze")
        return
    
    print(f"✅ Successful tests: {len(successful)}")
    print(f"❌ Failed tests: {len(failed)}")
    
    # Sort by computation time
    sorted_results = sorted(
        successful.items(),
        key=lambda x: x[1]['computation_time']
    )
    
    # Performance ranking
    print("\n🏆 Performance Ranking (faster is better):")
    for i, (compression, data) in enumerate(sorted_results, 1):
        time_s = data['computation_time']
        memory_mb = data.get('final_memory_mb', 0)
        spill_gb = data.get('max_spill_gb', 0)
        
        if i == 1:
            emoji = "🥇"
        elif i == 2:
            emoji = "🥈"
        elif i == 3:
            emoji = "🥉"
        else:
            emoji = "  "
            
        print(f"{emoji} {i}. {compression:8}: {time_s:6.2f}s | Mem: {memory_mb:6.1f}MB | Spill: {spill_gb:5.1f}GB")
    
    # Relative performance
    if sorted_results:
        fastest_time = sorted_results[0][1]['computation_time']
        print(f"\n⚡ Relative Performance (vs fastest):")
        for compression, data in sorted_results:
            relative = data['computation_time'] / fastest_time
            print(f"   {compression:8}: {relative:5.2f}x")
    
    # Memory efficiency analysis
    print(f"\n💾 Memory Efficiency Analysis:")
    for compression, data in successful.items():
        memory_mb = data.get('final_memory_mb', 0)
        spill_gb = data.get('max_spill_gb', 0)
        
        # Calculate efficiency score (lower is better)
        total_storage_mb = memory_mb + (spill_gb * 1024)
        efficiency_score = total_storage_mb / max(data['computation_time'], 0.1)
        
        print(f"   {compression:8}: Total storage={total_storage_mb:6.0f}MB, Efficiency={efficiency_score:6.0f}MB/s")
    
    # Failed tests summary
    if failed:
        print(f"\n❌ Failed Tests:")
        for compression, data in failed.items():
            error = data.get('error', 'Unknown error')
            print(f"   {compression:8}: {error}")
    
    return {
        'successful_count': len(successful),
        'failed_count': len(failed),
        'fastest': sorted_results[0] if sorted_results else None,
        'slowest': sorted_results[-1] if sorted_results else None
    }

# Analyze our compression results
if 'compression_results' in locals() and compression_results:
    compression_analysis = analyze_compression_results(compression_results)
else:
    print("⚠️  No compression results available for analysis")
    compression_analysis = None

## Memory Management Best Practices

Based on our experiments, let's summarize the key best practices for memory management in dask_setup.

In [ ]:
def print_memory_best_practices():
    """Print comprehensive memory management best practices."""
    
    print("\n" + "=" * 60)
    print("💡 MEMORY MANAGEMENT BEST PRACTICES")
    print("=" * 60)
    
    print("""
🎯 COMPRESSION ALGORITHM SELECTION:

   • lz4: Fastest compression, minimal CPU overhead
     → Best for: CPU-bound workloads, fast storage
     → Trade-off: Larger spill files, more disk I/O
     
   • zstd: Better compression ratio, moderate CPU cost
     → Best for: Limited disk space, network filesystems
     → Trade-off: Slightly slower but more space-efficient
     
   • snappy: Very fast, Google-developed
     → Best for: Real-time processing, streaming workloads
     → Trade-off: Good balance of speed vs compression
     
   • auto: Let Dask choose based on data characteristics
     → Best for: Variable data types, general use
     → Trade-off: May not be optimal for specific cases

⚙️ SPILLING THRESHOLD OPTIMIZATION:

   Memory Pressure Levels:
   • memory.target (0.60-0.80): Start spilling early on constrained systems
   • memory.spill (0.70-0.90): Main spilling threshold 
   • memory.pause (0.85-0.95): Pause new tasks to prevent OOM
   • memory.terminate (0.95-0.99): Last resort worker termination
   
   Recommended Configurations:
   
   Memory-Constrained (< 32 GB total):
   → target=0.60, spill=0.70, pause=0.85, terminate=0.95
   
   Standard Systems (32-128 GB):
   → target=0.75, spill=0.85, pause=0.92, terminate=0.98
   
   High-Memory Systems (> 128 GB):
   → target=0.80, spill=0.90, pause=0.95, terminate=0.99

💾 WORKER CONFIGURATION STRATEGIES:

   Few Large Workers (Recommended for most cases):
   • max_workers = cores // 4
   • More memory per worker = less spilling
   • Better for large intermediate results
   
   Many Small Workers (Special cases only):
   • max_workers = cores
   • Use for embarrassingly parallel tasks
   • Risk: More memory pressure per worker

🚨 HPC STORAGE CONSIDERATIONS:

   Fast Local Storage ($PBS_JOBFS):
   • Always preferred for spill directories
   • Monitor quota usage in job scripts
   • Clean up after job completion
   
   Shared Filesystems (lustre, gpfs):
   • Avoid for spilling if possible
   • Use compression to reduce I/O load
   • Consider network bandwidth limits
   
   Disk Quota Management:
   • Set conservative spill limits
   • Monitor disk usage during jobs
   • Plan for peak usage in resource requests

🔧 CONFIGURATION EXAMPLES:

   # Memory-constrained HPC node
   client, cluster, temp_dir = setup_dask_client(
       workload_type="cpu",
       max_workers=1,
       reserve_mem_gb=8.0  # Leave more for system
   )
   
   # High-performance setup
   client, cluster, temp_dir = setup_dask_client(
       workload_type="cpu", 
       max_workers=4,
       reserve_mem_gb=20.0  # Standard reservation
   )
   
   # I/O-intensive with spilling
   client, cluster, temp_dir = setup_dask_client(
       workload_type="io",
       max_workers=2,
       reserve_mem_gb=30.0  # Extra for I/O buffers
   )

📊 MONITORING RECOMMENDATIONS:

   Real-time Monitoring:
   • Use Dask dashboard for live memory tracking
   • Monitor spill directory usage
   • Watch for worker restarts (sign of memory issues)
   
   Performance Metrics:
   • Memory efficiency: memory_used / (memory_used + spill_used)
   • Spill frequency: Number of spill events per computation
   • I/O wait: Time spent reading/writing spill files
   
   Warning Signs:
   • Frequent worker restarts
   • High spill usage relative to memory
   • Long I/O wait times
   • System memory exhaustion
""")

# Display the best practices
print_memory_best_practices()

## Personalized Recommendations

Based on your system and our test results, here are specific recommendations:

In [ ]:
def generate_memory_recommendations():
    """Generate personalized memory management recommendations."""
    
    print("\n" + "=" * 60)
    print("🎯 PERSONALIZED RECOMMENDATIONS")
    print("=" * 60)
    
    # System analysis
    if PSUTIL_AVAILABLE:
        mem = psutil.virtual_memory()
        total_gb = mem.total / (1024**3)
        available_gb = mem.available / (1024**3)
        cpu_count = psutil.cpu_count(logical=False)
        
        print(f"\n📊 Your System:")
        print(f"   Total Memory: {total_gb:.1f} GB")
        print(f"   Available: {available_gb:.1f} GB ({mem.percent:.1f}% used)")
        print(f"   CPU Cores: {cpu_count}")
        
        # Memory category
        if total_gb < 16:
            memory_category = "Low Memory"
            recommended_workers = 1
            recommended_reserve = min(4.0, total_gb * 0.3)
        elif total_gb < 64:
            memory_category = "Standard Memory"
            recommended_workers = min(4, cpu_count // 2)
            recommended_reserve = min(20.0, total_gb * 0.25)
        else:
            memory_category = "High Memory"
            recommended_workers = min(8, cpu_count // 2)
            recommended_reserve = min(40.0, total_gb * 0.2)
        
        print(f"   Category: {memory_category}")
        
    else:
        print(f"\n📊 System information not available")
        recommended_workers = 2
        recommended_reserve = 20.0
        memory_category = "Unknown"
    
    # Compression recommendations based on test results
    compression_recommendation = "lz4"  # Default
    
    if 'compression_analysis' in globals() and compression_analysis:
        if compression_analysis.get('fastest'):
            fastest_compression = compression_analysis['fastest'][0]
            print(f"\n⚡ Based on your tests, {fastest_compression} was fastest")
            compression_recommendation = fastest_compression
        else:
            print(f"\n⚠️  Could not determine optimal compression from tests")
    else:
        print(f"\n💡 No compression test data available - using defaults")
    
    # Generate specific recommendations
    print(f"\n🔧 RECOMMENDED CONFIGURATION:")
    
    print(f"\n   Basic Setup:")
    print(f"   ```python")
    print(f"   client, cluster, temp_dir = setup_dask_client(")
    print(f"       workload_type='cpu',")
    print(f"       max_workers={recommended_workers},")
    print(f"       reserve_mem_gb={recommended_reserve:.1f}")
    print(f"   )")
    print(f"   ```")
    
    # Workload-specific recommendations
    print(f"\n   Workload-Specific Configurations:")
    
    print(f"\n   Memory-Intensive Tasks:")
    print(f"   ```python")
    print(f"   # Use fewer workers with more memory each")
    print(f"   client, cluster, temp_dir = setup_dask_client(")
    print(f"       workload_type='cpu',")
    print(f"       max_workers={max(1, recommended_workers//2)},")
    print(f"       reserve_mem_gb={recommended_reserve * 0.8:.1f}")
    print(f"   )")
    print(f"   ```")
    
    print(f"\n   I/O-Heavy Tasks:")
    print(f"   ```python")
    print(f"   # More workers, optimized for I/O")
    print(f"   client, cluster, temp_dir = setup_dask_client(")
    print(f"       workload_type='io',")
    print(f"       max_workers={min(8, recommended_workers * 2)},")
    print(f"       reserve_mem_gb={recommended_reserve:.1f}")
    print(f"   )")
    print(f"   ```")
    
    # Memory monitoring recommendations
    print(f"\n💾 MONITORING RECOMMENDATIONS:")
    print(f"   • Always use dashboard=True for development")
    print(f"   • Monitor spill usage in {os.environ.get('TMPDIR', '/tmp')}")
    print(f"   • Watch for worker memory above 80%")
    print(f"   • Set up alerts for disk space in spill directory")
    
    if memory_category == "Low Memory":
        print(f"\n⚠️  LOW MEMORY SYSTEM TIPS:")
        print(f"   • Use aggressive spilling (target=0.60)")
        print(f"   • Process data in smaller batches")
        print(f"   • Consider using swap space")
        print(f"   • Monitor system memory constantly")
    
    return {
        'recommended_workers': recommended_workers,
        'recommended_reserve_gb': recommended_reserve,
        'memory_category': memory_category,
        'compression_recommendation': compression_recommendation
    }

# Generate personalized recommendations
recommendations = generate_memory_recommendations()

## Conclusion

You've now learned advanced memory management techniques with dask_setup! Here's what we covered:

### Key Takeaways:
1. **Memory Monitoring**: Real-time tracking of memory usage and spill patterns
2. **Compression Impact**: Different algorithms have trade-offs between speed and space
3. **Spill Management**: Understanding when and why workers spill to disk
4. **System Optimization**: Tailoring configuration to your hardware constraints
5. **HPC Considerations**: Leveraging fast local storage and managing quotas

### Best Practices Summary:
- **Monitor actively**: Use dashboards and memory tracking functions
- **Configure appropriately**: Match worker count and memory to your system
- **Choose compression wisely**: Balance speed vs storage efficiency
- **Plan for spilling**: Ensure adequate fast storage for temporary files
- **Test your workload**: Benchmark different configurations with your actual data

### Next Steps:
- Apply these memory management techniques to your real workloads
- Experiment with different spilling thresholds for your use cases
- Set up monitoring for production environments
- Consider HPC-specific optimizations for cluster environments
- Integrate memory-aware chunking strategies

### Troubleshooting Guide:
- **Frequent worker restarts**: Lower memory thresholds or reduce workers
- **Slow performance**: Check if excessive spilling is causing I/O bottlenecks
- **Disk space issues**: Monitor spill directory usage and clean up regularly
- **Memory leaks**: Use memory timeline analysis to identify problematic operations

Happy memory-efficient computing with dask_setup! 🧠💾